# Ensemble Learning Dashboard: Sri Lankan Used Car Price Prediction

**Generative AI Disclosure:** In accordance with course guidelines, Generative AI (Gemini) was used to assist with Python data-cleaning boilerplate, model comparison structure, and dashboard UI layout.

In [1]:
import pandas as pd

df = pd.read_csv("sl_cars.csv")
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nData Types:\n", df.dtypes)
print("\nMissing values:\n", df.isnull().sum())
print("\nFirst 5 rows:")
df.head(5)

Shape: (9788, 17)

Columns: ['Unnamed: 0', 'Brand', 'Model', 'YOM', 'Engine (cc)', 'Gear', 'Fuel Type', 'Millage(KM)', 'Town', 'Date', 'Leasing', 'Condition', 'AIR CONDITION', 'POWER STEERING', 'POWER MIRROR', 'POWER WINDOW', 'Price']

Data Types:
 Unnamed: 0          int64
Brand              object
Model              object
YOM                 int64
Engine (cc)       float64
Gear               object
Fuel Type          object
Millage(KM)       float64
Town               object
Date               object
Leasing            object
Condition          object
AIR CONDITION      object
POWER STEERING     object
POWER MIRROR       object
POWER WINDOW       object
Price             float64
dtype: object

Missing values:
 Unnamed: 0        0
Brand             0
Model             0
YOM               0
Engine (cc)       0
Gear              0
Fuel Type         0
Millage(KM)       0
Town              0
Date              0
Leasing           0
Condition         0
AIR CONDITION     0
POWER STEERING   

,Unnamed: 0,Brand,Model,YOM,Engine (cc),Gear,Fuel Type,Millage(KM),Town,Date,Leasing,Condition,AIR CONDITION,POWER STEERING,POWER MIRROR,POWER WINDOW,Price
0,0,AUDI,A1,2016,990.0,Automatic,Petrol,99000.0,Gampaha,2025-02-05,No Leasing,USED,Available,Available,Available,Available,100.0
1,1,AUDI,A1,2017,1000.0,Automatic,Petrol,88000.0,Colombo,2025-01-14,No Leasing,USED,Available,Available,Available,Available,97.0
2,2,AUDI,A1,2018,1000.0,Automatic,Petrol,77000.0,Dehiwala-Mount-Lavinia,2025-01-23,No Leasing,USED,Available,Available,Available,Available,98.5
3,3,AUDI,A1,2017,1000.0,Automatic,Petrol,88000.0,Negombo,2024-12-21,No Leasing,USED,Available,Available,Available,Available,107.0
4,4,AUDI,A1,2017,1000.0,Automatic,Petrol,88000.0,Colombo,2024-12-21,No Leasing,USED,Available,Available,Available,Available,99.5


In [2]:
import pandas as pd
import numpy as np

# 1. Load & drop unnecessary columns
df = pd.read_csv("sl_cars.csv")
df = df.drop(columns=["Unnamed: 0", "Date"], errors="ignore")
df = df.rename(columns={"Millage(KM)": "Mileage_KM", "Engine (cc)": "Engine_CC"})

# 2. Strip extra spaces from text columns
cat_raw = ["Brand", "Model", "Gear", "Fuel Type", "Town", "Leasing", "Condition",
           "AIR CONDITION", "POWER STEERING", "POWER MIRROR", "POWER WINDOW"]
for col in cat_raw:
    df[col] = df[col].astype(str).str.strip()

# 3. Convert Price from LKR Lakhs -> LKR Millions (e.g., 100.0 Lakhs = 10.0M LKR)
df["Price_LKR_M"] = df["Price"] / 10.0
df = df.drop(columns=["Price"])

# 4. Convert binary equipment columns to 1 / 0
binary_cols = ["AIR CONDITION", "POWER STEERING", "POWER MIRROR", "POWER WINDOW"]
for col in binary_cols:
    df[col] = (df[col].str.lower() == "available").astype(int)

# 5. Remove duplicates and unrealistic scraped outliers
before_rows = len(df)
df = df.drop_duplicates()
df = df[
    (df["Price_LKR_M"].between(0.5, 100.0)) &   # LKR 500k to 100M
    (df["YOM"].between(1980, 2025)) &
    (df["Engine_CC"].between(500, 6000)) &
    (df["Mileage_KM"].between(0, 500000))
].copy()

# 6. Group rare Towns (keep top 25) and very rare Models (< 5 listings) so encoding stays clean
top_towns = df["Town"].value_counts().nlargest(25).index
df["Town"] = df["Town"].where(df["Town"].isin(top_towns), "Other")

model_counts = df["Model"].value_counts()
valid_models = model_counts[model_counts >= 5].index
df["Model"] = df["Model"].where(df["Model"].isin(valid_models), "Other")

print(f"Rows before cleaning: {before_rows} -> After cleaning: {len(df)}")
print(f"Unique Brands: {df['Brand'].nunique()} | Unique Models: {df['Model'].nunique()} | Unique Towns: {df['Town'].nunique()}")
print("\nPrice (LKR Millions) summary:")
print(df["Price_LKR_M"].describe().round(2))

Rows before cleaning: 9788 -> After cleaning: 9427
Unique Brands: 49 | Unique Models: 290 | Unique Towns: 26

Price (LKR Millions) summary:
count    9427.00
mean        5.40
std         4.37
min         0.50
25%         2.67
50%         4.38
75%         6.98
max        79.00
Name: Price_LKR_M, dtype: float64


In [3]:
import os
import json
import time
import shutil
import joblib
import sklearn
import xgboost
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

# Feature groups
cat_cols = ["Brand", "Model", "Gear", "Fuel Type", "Town", "Leasing", "Condition"]
num_cols = ["YOM", "Engine_CC", "Mileage_KM"]
bin_cols = ["AIR CONDITION", "POWER STEERING", "POWER MIRROR", "POWER WINDOW"]

X = df[cat_cols + num_cols + bin_cols]
y = df["Price_LKR_M"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
        ("num", StandardScaler(), num_cols),
        ("bin", "passthrough", bin_cols),
    ]
)

# 1. Bagging: Random Forest
rf_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestRegressor(n_estimators=150, max_depth=16, min_samples_leaf=2, random_state=42, n_jobs=-1))
])

# 2a. Boosting: Gradient Boosting
gbr_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", GradientBoostingRegressor(n_estimators=200, learning_rate=0.07, max_depth=5, random_state=42))
])

# 2b. Boosting: XGBoost
xgb_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", XGBRegressor(n_estimators=300, learning_rate=0.06, max_depth=6, subsample=0.85, colsample_bytree=0.85, random_state=42, n_jobs=-1))
])

# 3. Stacking: RF + XGBoost + GBR -> Ridge Meta-Learner
stack_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", StackingRegressor(
        estimators=[
            ("rf", RandomForestRegressor(n_estimators=100, max_depth=14, min_samples_leaf=2, random_state=42, n_jobs=-1)),
            ("xgb", XGBRegressor(n_estimators=250, learning_rate=0.06, max_depth=6, subsample=0.85, random_state=42, n_jobs=-1)),
            ("gbr", GradientBoostingRegressor(n_estimators=150, learning_rate=0.07, max_depth=5, random_state=42)),
        ],
        final_estimator=Ridge(alpha=1.0),
        cv=5,
        n_jobs=-1
    ))
])

models = {
    "Bagging (Random Forest)": rf_pipe,
    "Boosting (Gradient Boosting)": gbr_pipe,
    "Boosting (XGBoost)": xgb_pipe,
    "Stacking (RF + XGB + GBR)": stack_pipe,
}

results = {}
pred_df = pd.DataFrame({"Actual": y_test.values})

for name, pipe in models.items():
    t0 = time.time()
    pipe.fit(X_train, y_train)
    train_time = round(time.time() - t0, 2)
    preds = pipe.predict(X_test)
    pred_df[name] = np.round(preds, 3)
    results[name] = {
        "RMSE (LKR M)": round(float(np.sqrt(mean_squared_error(y_test, preds))), 3),
        "MAE (LKR M)": round(float(mean_absolute_error(y_test, preds)), 3),
        "R2 Score": round(float(r2_score(y_test, preds)), 4),
        "Train Time (s)": train_time
    }
    print(f"Finished {name} in {train_time}s")

# 4. Hybrid Model: RF + ANN Residual Learning Hybrid
t0 = time.time()
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

hybrid_rf = RandomForestRegressor(n_estimators=150, max_depth=16, min_samples_leaf=2, random_state=42, n_jobs=-1)
hybrid_rf.fit(X_train_proc, y_train)

train_residuals = y_train - hybrid_rf.predict(X_train_proc)
hybrid_ann = MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=300, early_stopping=True, random_state=42)
hybrid_ann.fit(X_train_proc, train_residuals)
hybrid_time = round(time.time() - t0, 2)

hybrid_preds = hybrid_rf.predict(X_test_proc) + hybrid_ann.predict(X_test_proc)
pred_df["Hybrid (RF + ANN)"] = np.round(hybrid_preds, 3)
results["Hybrid (RF + ANN)"] = {
    "RMSE (LKR M)": round(float(np.sqrt(mean_squared_error(y_test, hybrid_preds))), 3),
    "MAE (LKR M)": round(float(mean_absolute_error(y_test, hybrid_preds)), 3),
    "R2 Score": round(float(r2_score(y_test, hybrid_preds)), 4),
    "Train Time (s)": hybrid_time
}
print(f"Finished Hybrid (RF + ANN) in {hybrid_time}s")

# Display sorted comparison table
metrics_df = pd.DataFrame(results).T.sort_values("RMSE (LKR M)")
print("\n=== MODEL COMPARISON RESULTS ===")
display(metrics_df)

# Save artifacts (compressed to stay well under GitHub's 100MB limit)
os.makedirs("artifacts", exist_ok=True)

# Store UI dropdown options so Brand -> Model filters dynamically in Streamlit!
brand_to_models = (
    df.groupby("Brand")["Model"]
    .apply(lambda s: sorted(s.unique().tolist()))
    .to_dict()
)
ui_options = {
    "brand_to_models": brand_to_models,
    "gears": sorted(df["Gear"].unique().tolist()),
    "fuels": sorted(df["Fuel Type"].unique().tolist()),
    "towns": sorted(df["Town"].unique().tolist()),
    "leasings": sorted(df["Leasing"].unique().tolist()),
    "conditions": sorted(df["Condition"].unique().tolist()),
}

bundle = {
    "models": models,
    "hybrid_prep": preprocessor,
    "hybrid_rf": hybrid_rf,
    "hybrid_ann": hybrid_ann,
}

joblib.dump(bundle, "artifacts/models_bundle.joblib", compress=3)
pred_df.to_csv("artifacts/test_predictions.csv", index=False)
with open("artifacts/metrics.json", "w") as f:
    json.dump(results, f, indent=2)
with open("artifacts/ui_options.json", "w") as f:
    json.dump(ui_options, f, indent=2)

# Check bundle size & print exact package versions for Streamlit Cloud
size_mb = os.path.getsize("artifacts/models_bundle.joblib") / (1024 * 1024)
print(f"\nSaved models_bundle.joblib ({size_mb:.2f} MB)")
print(f"VERSIONS -> scikit-learn=={sklearn.__version__} | xgboost=={xgboost.__version__} | pandas=={pd.__version__} | numpy=={np.__version__} | joblib=={joblib.__version__}")

# Zip artifacts folder for easy download
shutil.make_archive("artifacts", "zip", "artifacts")
print("Created artifacts.zip ready for download!")

Finished Bagging (Random Forest) in 22.58s
Finished Boosting (Gradient Boosting) in 23.47s
Finished Boosting (XGBoost) in 1.61s
Finished Stacking (RF + XGB + GBR) in 145.64s
Finished Hybrid (RF + ANN) in 16.62s

=== MODEL COMPARISON RESULTS ===


,RMSE (LKR M),MAE (LKR M),R2 Score,Train Time (s)
Hybrid (RF + ANN),1.219,0.594,0.9143,16.62
Bagging (Random Forest),1.241,0.576,0.9112,22.58
Stacking (RF + XGB + GBR),1.262,0.603,0.9081,145.64
Boosting (XGBoost),1.285,0.611,0.9048,1.61
Boosting (Gradient Boosting),1.307,0.664,0.9015,23.47



Saved models_bundle.joblib (15.95 MB)
VERSIONS -> scikit-learn==1.6.1 | xgboost==3.4.1 | pandas==2.2.3 | numpy==2.4.6 | joblib==1.6.0
Created artifacts.zip ready for download!


## Analytical Interpretation: Why the Models Behaved This Way

1. **Hybrid (RF + ANN) — Best Overall (`RMSE: 1.219M LKR`, `R²: 0.9143`, `16.62s`):**
   Combining Random Forest with a residual-learning Neural Network produced the lowest error and highest $R^2$. Random Forest excels at sharp, high-cardinality categorical partitions in Sri Lankan vehicle listings (e.g., *Toyota Premio* vs. *Suzuki Alto*, *Hybrid* vs. *Petrol*), while the MLPRegressor learns the smooth continuous depreciation residuals driven by Year of Manufacture (YOM), Mileage, and Engine CC—fixing Random Forest's step-function limitation.

2. **Bagging (Random Forest) — Strongest Single Ensemble & Lowest MAE (`RMSE: 1.241M LKR`, `MAE: 0.576M LKR`, `R²: 0.9112`):**
   Used-car listings scraped from Sri Lankan marketplaces contain noisy outliers (negotiable prices, condition variance, and rare trims). Random Forest's bootstrap averaging naturally suppresses high-variance noise, giving it the best median-like accuracy (`MAE = 0.576M LKR`) and beating both boosting models.

3. **Boosting (XGBoost & Gradient Boosting) — Fastest Training (`1.61s`), Slightly Higher Error (`RMSE: 1.285M` & `1.307M`):**
   Sequential boosting fits each tree to prior errors. On noisy real-world scraped car prices, boosting can over-adjust toward listing outliers, leading to slightly higher test RMSE than bagging. However, XGBoost trained in just **1.61 seconds**—making it the best speed-to-accuracy trade-off.

4. **Stacking (RF + XGB + GBR → Ridge) — Balanced Generalization, Highest Compute Cost (`RMSE: 1.262M LKR`, `R²: 0.9081`, `145.64s`):**
   Stacking improved over both standalone boosting models by blending their predictions with Random Forest via 5-fold cross-validation, but at **145.64 seconds** (nearly 9× slower than the RF+ANN hybrid) without beating the hybrid's accuracy.